# Random Forest — APK Files
Malware detection using static analysis features from EMBER2024.  
File type: `APK` (Android) executables  
Model: Random Forest

## 1. Imports

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, log_loss, roc_auc_score, roc_curve,
    average_precision_score
)

## 2. Config

In [ ]:
# ---- CHANGE THIS ----
BASE_PATH = r"C:\Users\makis\ember_data\apk_clean"
# ---------------------

DIM          = 2568
RANDOM_STATE = 13

# EMBER2024 feature group offsets
GENERAL_START,  GENERAL_END  = 0,    7      # GeneralFileInfo       (7)
HIST_START,     HIST_END     = 7,    263    # ByteHistogram         (256)
ENTROPY_START,  ENTROPY_END  = 263,  519    # ByteEntropyHistogram  (256)
STRING_START,   STRING_END   = 519,  696    # StringExtractor       (177)
HEADER_START,   HEADER_END   = 696,  770    # HeaderFileInfo        (74)
SECTION_START,  SECTION_END  = 770,  994    # SectionInfo           (224)
IMPORT_START,   IMPORT_END   = 994,  2276   # Imports               (1282)
EXPORT_START,   EXPORT_END   = 2276, 2405   # Exports               (129)
DATADIR_START,  DATADIR_END  = 2405, 2439   # DataDirectories       (34)
RICH_START,     RICH_END     = 2439, 2472   # RichHeader            (33)
AUTH_START,     AUTH_END     = 2472, 2480   # AuthenticodeSignature (8)
PEWARN_START,   PEWARN_END   = 2480, 2568   # PEFormatWarnings      (88)
DOS_START,      DOS_END      = 753,  770    # DOS Header (inside HeaderFileInfo)

## 3. Load Data

In [ ]:
X_train_full = np.memmap(os.path.join(BASE_PATH, "X_train.dat"), dtype=np.float32, mode="r").reshape(-1, DIM)
y_train      = np.memmap(os.path.join(BASE_PATH, "y_train.dat"), dtype=np.int32,   mode="r")

X_test_full  = np.memmap(os.path.join(BASE_PATH, "X_test.dat"),  dtype=np.float32, mode="r").reshape(-1, DIM)
y_test       = np.memmap(os.path.join(BASE_PATH, "y_test.dat"),  dtype=np.int32,   mode="r")

print(f"Train: {X_train_full.shape[0]:,} samples | Test: {X_test_full.shape[0]:,} | Features: {DIM}")
print("Train labels:", dict(pd.Series(y_train).value_counts()))
print("Test  labels:", dict(pd.Series(y_test).value_counts()))

## 4. Helpers

In [ ]:
def evaluate(model, X, y, label=""):
    t0 = time.perf_counter()
    proba = model.predict_proba(X)[:, 1]
    pred  = (proba >= 0.5).astype(int)
    inf_time = time.perf_counter() - t0

    acc  = accuracy_score(y, pred)
    prec = precision_score(y, pred, zero_division=0)
    rec  = recall_score(y, pred, zero_division=0)
    f1   = f1_score(y, pred, zero_division=0)
    ll   = log_loss(y, proba)
    roc  = roc_auc_score(y, proba)
    pr   = average_precision_score(y, proba)
    cm   = confusion_matrix(y, pred)

    print(f"\n=== {label} ===")
    print(f"Inference time : {inf_time:.4f} s")
    print(f"Accuracy       : {acc:.6f}")
    print(f"Precision      : {prec:.6f}")
    print(f"Recall         : {rec:.6f}")
    print(f"F1             : {f1:.6f}")
    print(f"Log Loss       : {ll:.6f}")
    print(f"ROC-AUC        : {roc:.6f}")
    print(f"PR-AUC         : {pr:.6f}")
    print(f"Confusion Matrix:\n{cm}")

    fpr, tpr, _ = roc_curve(y, proba)
    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f"{label} (AUC={roc:.4f})")
    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve — {label}")
    plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
    return {"label": label, "accuracy": acc, "f1": f1, "roc_auc": roc, "pr_auc": pr}

def run_experiment(X_tr, X_te, y_tr, y_te, model, label):
    print(f"\nFeatures: {X_tr.shape[1]} cols")
    t0 = time.perf_counter()
    model.fit(X_tr, y_tr)
    print(f"Training time: {time.perf_counter() - t0:.2f} s")
    return evaluate(model, X_te, y_te, label)

> **Note:** APK files do not follow the PE format. PE-specific feature groups
> (Imports, Exports, Headers, Sections, DataDirectories, RichHeader, Authenticode)
> are all-zero for APK samples. Only byte-level and string features are evaluated here.

## 5. Baseline Model

In [ ]:
rf_baseline = RandomForestClassifier(
    n_estimators=100, max_features="sqrt", n_jobs=-1,
    random_state=RANDOM_STATE, class_weight="balanced_subsample"
)

print("Training baseline Random Forest...")
t0 = time.perf_counter()
rf_baseline.fit(X_train_full, y_train)
print(f"Training time: {time.perf_counter() - t0:.2f} s")

evaluate(rf_baseline, X_test_full, y_test, "RF Baseline — Full Features")

## 6. Hyperparameter Tuning
### Phase 1 — n_estimators & max_features

In [ ]:
TRAIN_SIZE = 50_000
EVAL_SIZE  = 10_000

X_sub_train = X_train_full[:TRAIN_SIZE]
y_sub_train = y_train[:TRAIN_SIZE]
X_eval      = X_train_full[TRAIN_SIZE:TRAIN_SIZE + EVAL_SIZE]
y_eval      = y_train[TRAIN_SIZE:TRAIN_SIZE + EVAL_SIZE]

results = []
for n_est in [100, 200, 400]:
    for mf in ["sqrt", 0.1, 0.2, 0.3]:
        rf = RandomForestClassifier(n_estimators=n_est, max_features=mf,
                                     n_jobs=-1, random_state=RANDOM_STATE,
                                     class_weight="balanced_subsample")
        t0 = time.perf_counter()
        rf.fit(X_sub_train, y_sub_train)
        train_time = time.perf_counter() - t0
        proba = rf.predict_proba(X_eval)[:, 1]
        results.append({"n_estimators": n_est, "max_features": mf,
                        "roc_auc": roc_auc_score(y_eval, proba),
                        "pr_auc":  average_precision_score(y_eval, proba),
                        "train_time_s": train_time})

df_p1 = pd.DataFrame(results).sort_values("pr_auc", ascending=False)
print(df_p1.head(8).to_string(index=False))

### Phase 2 — Depth & leaf params

In [ ]:
results2 = []
for max_depth in [20, 40, None]:
    for min_leaf in [1, 2, 4]:
        for min_split in [2, 5]:
            rf = RandomForestClassifier(
                n_estimators=400, max_features=0.2,
                max_depth=max_depth, min_samples_leaf=min_leaf,
                min_samples_split=min_split,
                n_jobs=-1, random_state=RANDOM_STATE,
                class_weight="balanced_subsample"
            )
            t0 = time.perf_counter()
            rf.fit(X_sub_train, y_sub_train)
            train_time = time.perf_counter() - t0
            proba = rf.predict_proba(X_eval)[:, 1]
            results2.append({"max_depth": max_depth, "min_leaf": min_leaf,
                             "min_split": min_split,
                             "roc_auc": roc_auc_score(y_eval, proba),
                             "pr_auc":  average_precision_score(y_eval, proba),
                             "train_time_s": train_time})

df_p2 = pd.DataFrame(results2).sort_values("pr_auc", ascending=False)
print(df_p2.head(8).to_string(index=False))

## 7. Tuned Model — Full Training & Evaluation

In [ ]:
# Best params found via grid search
BEST_PARAMS = {
    "n_estimators":  200,
    "max_features":  0.2,
    "class_weight":  "balanced_subsample",
}

rf_tuned = RandomForestClassifier(
    **BEST_PARAMS, bootstrap=True, n_jobs=-1,
    random_state=RANDOM_STATE, verbose=1
)

print("Training tuned Random Forest on full training set...")
t0 = time.perf_counter()
rf_tuned.fit(X_train_full, y_train)
print(f"Training time: {time.perf_counter() - t0:.2f} s")

evaluate(rf_tuned, X_test_full, y_test, "RF Tuned — Full Features")

## 8. Feature Group Experiments

For APK files, only byte-level and string features are non-zero.

In [ ]:
def get_rf():
    return RandomForestClassifier(**BEST_PARAMS, bootstrap=True,
                                   n_jobs=-1, random_state=RANDOM_STATE, verbose=1)

def run_exp(X_tr, X_te, label):
    return run_experiment(X_tr, X_te, y_train, y_test, get_rf(), label)

In [ ]:
# ByteHistogram + ByteEntropyHistogram + GeneralFileInfo
run_exp(X_train_full[:, GENERAL_START:ENTROPY_END],
        X_test_full[:,  GENERAL_START:ENTROPY_END],
        "RF — ByteHistogram + ByteEntropyHistogram + GeneralFileInfo")

In [ ]:
# Strings only
run_exp(X_train_full[:, STRING_START:STRING_END],
        X_test_full[:,  STRING_START:STRING_END],
        "RF — Strings only")

In [ ]:
# Full features (all groups including zero PE groups)
run_exp(X_train_full, X_test_full, "RF — All Features")